#  Advanced Speech-to-Text

**faster-whisper · word timestamps · confidence scores · speaker diarization · multi-format export**


## Step 0 — Install dependencies



In [5]:
!pip install -q faster-whisper librosa soundfile pandas
print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 81.8 MB/s eta 0:00:00
Dependencies installed.


## Step 1 — Check your hardware

In [6]:
import torch
if torch.cuda.is_available():
    print("GPU available:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Go to Runtime > Change runtime type > T4 GPU for ~10x speedup.")

No GPU. Go to Runtime > Change runtime type > T4 GPU for ~10x speedup.


## Step 2 — The transcription engine





In [7]:
import os, json, math
from dataclasses import dataclass, field, asdict
from typing import List, Optional


@dataclass
class Word:
    text: str; start: float; end: float; probability: float

@dataclass
class Segment:
    text: str; start: float; end: float; confidence: float
    speaker: Optional[str] = None
    words: List[Word] = field(default_factory=list)

@dataclass
class Transcript:
    text: str; language: str; duration: float
    segments: List[Segment] = field(default_factory=list)


class AdvancedTranscriber:
    """Reusable transcription engine (Colab edition)."""

    def __init__(self, model_size="base", device="auto", compute_type="auto"):
        from faster_whisper import WhisperModel
        if device == "auto" or compute_type == "auto":
            has_gpu = torch.cuda.is_available()
            if device == "auto":
                device = "cuda" if has_gpu else "cpu"
            if compute_type == "auto":
                compute_type = "float16" if has_gpu else "int8"
        print(f"Loading '{model_size}' on {device} ({compute_type})...")
        self.model = WhisperModel(model_size, device=device, compute_type=compute_type)
        print("Ready.")

    @staticmethod
    def preprocess_audio(path, target_sr=16000):
        """Load any file -> mono, 16kHz, peak-normalized float32 array."""
        import librosa, numpy as np
        audio, _ = librosa.load(path, sr=target_sr, mono=True)
        peak = np.abs(audio).max()
        if peak > 0:
            audio = audio / peak * 0.95
        return audio.astype(np.float32)

    def transcribe(self, path, language=None, preprocess=True, translate=False):
        audio_input = self.preprocess_audio(path) if preprocess else path
        segments_iter, info = self.model.transcribe(
            audio_input,
            language=language,
            task="translate" if translate else "transcribe",
            beam_size=5,
            word_timestamps=True,
            vad_filter=True,
            vad_parameters={"min_silence_duration_ms": 500},
        )
        segments, parts = [], []
        for seg in segments_iter:
            conf = math.exp(seg.avg_logprob) if seg.avg_logprob else 0.0
            words = [Word(w.word.strip(), round(w.start,3), round(w.end,3),
                          round(w.probability,3)) for w in (seg.words or [])]
            segments.append(Segment(seg.text.strip(), round(seg.start,3),
                                    round(seg.end,3), round(conf,3), words=words))
            parts.append(seg.text.strip())
        return Transcript(" ".join(parts), info.language,
                          round(info.duration,2), segments)

    # ---- export ----
    @staticmethod
    def _fmt_time(s, sep=","):
        h=int(s//3600); m=int((s%3600)//60); sec=int(s%60); ms=int((s%1)*1000)
        return f"{h:02d}:{m:02d}:{sec:02d}{sep}{ms:03d}"

    def export(self, t, basename, formats=("txt","srt","json")):
        written=[]
        for fmt in formats:
            out=f"{basename}.{fmt}"
            getattr(self, f"_write_{fmt}")(t, out)
            written.append(out); print("Wrote", out)
        return written

    def _write_txt(self, t, path):
        with open(path,"w",encoding="utf-8") as f:
            for s in t.segments:
                pre=f"[{s.speaker}] " if s.speaker else ""
                f.write(f"{pre}{s.text}\n")

    def _write_srt(self, t, path):
        with open(path,"w",encoding="utf-8") as f:
            for i,s in enumerate(t.segments,1):
                pre=f"{s.speaker}: " if s.speaker else ""
                f.write(f"{i}\n{self._fmt_time(s.start)} --> {self._fmt_time(s.end)}\n{pre}{s.text}\n\n")

    def _write_vtt(self, t, path):
        with open(path,"w",encoding="utf-8") as f:
            f.write("WEBVTT\n\n")
            for s in t.segments:
                pre=f"{s.speaker}: " if s.speaker else ""
                f.write(f"{self._fmt_time(s.start,'.')} --> {self._fmt_time(s.end,'.')}\n{pre}{s.text}\n\n")

    def _write_json(self, t, path):
        with open(path,"w",encoding="utf-8") as f:
            json.dump(asdict(t), f, ensure_ascii=False, indent=2)

    def _write_csv(self, t, path):
        import pandas as pd
        rows=[{"start":s.start,"end":s.end,"speaker":s.speaker or "",
               "confidence":s.confidence,"text":s.text} for s in t.segments]
        pd.DataFrame(rows).to_csv(path, index=False, encoding="utf-8")

    # ---- speaker diarization (optional) ----
    def add_speakers(self, transcript, audio_path, hf_token=None):
        try:
            from pyannote.audio import Pipeline
        except ImportError:
            print("pyannote not installed - run the diarization install cell first."); return transcript
        token = hf_token or os.environ.get("HF_TOKEN")
        if not token:
            print("No HF token - skipping diarization."); return transcript
        print("Running diarization...")
        pipe = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", use_auth_token=token)
        diar = pipe(audio_path)
        turns=[(t.start,t.end,spk) for t,_,spk in diar.itertracks(yield_label=True)]
        for seg in transcript.segments:
            best,bestov=None,0.0
            for ts,te,spk in turns:
                ov=min(seg.end,te)-max(seg.start,ts)
                if ov>bestov: bestov,best=ov,spk
            seg.speaker=best
        return transcript

print("AdvancedTranscriber loaded.")

AdvancedTranscriber loaded.


## Step 3 — Load a model



In [8]:
transcriber = AdvancedTranscriber(model_size="base")  # tiny|base|small|medium|large-v3

Loading 'base' on cpu (int8)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Ready.


## Step 4 — Upload an audio file
A file picker will appear. Choose any `.mp3`, `.wav`, `.m4a`, `.flac`, `.ogg`, or `.webm`.

In [9]:
from google.colab import files
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]
print("Uploaded:", audio_path)

Saving harvard.wav to harvard.wav
Uploaded: harvard.wav


## Step 5 — Transcribe

Set `language="hi"` to force Hindi, leave `None` to auto-detect, or set
`translate=True` to translate any language into English.

In [10]:
result = transcriber.transcribe(audio_path, language=None, translate=False)

print("=" * 60)
print(f"LANGUAGE: {result.language}   DURATION: {result.duration}s")
print("=" * 60)
print("\nFULL TEXT:\n", result.text, "\n")
print("SEGMENTS:")
for s in result.segments:
    tag = f"[{s.speaker}] " if s.speaker else ""
    print(f"  {s.start:6.1f}s  {tag}{s.text}   (conf {s.confidence:.2f})")

LANGUAGE: en   DURATION: 18.36s

FULL TEXT:
 The stale smell of old beer lingers. It takes heat to bring out the odor. A cold dip restores health and zest. A salt pickle tastes fine with ham. Tacos al pastor are my favorite. A zestful food is the hot cross bun. 

SEGMENTS:
     0.9s  The stale smell of old beer lingers.   (conf 0.75)
     4.2s  It takes heat to bring out the odor.   (conf 0.75)
     7.0s  A cold dip restores health and zest.   (conf 0.75)
    10.0s  A salt pickle tastes fine with ham.   (conf 0.75)
    12.7s  Tacos al pastor are my favorite.   (conf 0.75)
    15.1s  A zestful food is the hot cross bun.   (conf 0.75)


## Step 6 — View segments as a table

In [11]:
import pandas as pd
df = pd.DataFrame([{"start": s.start, "end": s.end,
                    "confidence": s.confidence, "text": s.text}
                   for s in result.segments])
df

,start,end,confidence,text
0,0.91,3.63,0.752,The stale smell of old beer lingers.
1,4.25,6.17,0.752,It takes heat to bring out the odor.
2,7.01,9.17,0.752,A cold dip restores health and zest.
3,9.97,11.99,0.752,A salt pickle tastes fine with ham.
4,12.65,14.31,0.752,Tacos al pastor are my favorite.
5,15.07,17.43,0.752,A zestful food is the hot cross bun.


## Step 7 — Export and download

Writes `.txt`, `.srt`, `.vtt`, `.json`, `.csv` and downloads them to your computer.

In [12]:
base = audio_path.rsplit(".", 1)[0]
files_written = transcriber.export(result, base, formats=["txt","srt","vtt","json","csv"])

from google.colab import files as colab_files
for f in files_written:
    colab_files.download(f)

Wrote harvard.txt
Wrote harvard.srt
Wrote harvard.vtt
Wrote harvard.json
Wrote harvard.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 8 (optional) — Speaker diarization: "who spoke when"

Labels each segment as SPEAKER_00, SPEAKER_01, ... Useful for meetings/interviews.

**One-time setup:**
1. Create a free token at https://huggingface.co/settings/tokens
2. Accept the terms at https://huggingface.co/pyannote/speaker-diarization-3.1
3. Paste your token below.

In [13]:
!pip install -q pyannote.audio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.7/893.7 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48

In [14]:
HF_TOKEN = ""  # <-- paste your Hugging Face token here

if HF_TOKEN:
    result = transcriber.add_speakers(result, audio_path, hf_token=HF_TOKEN)
    for s in result.segments:
        print(f"  {s.start:6.1f}s  [{s.speaker}]  {s.text}")
else:
    print("Add your HF_TOKEN above to enable diarization.")

Add your HF_TOKEN above to enable diarization.


## Step 9 (optional) — Record from your microphone in Colab

Colab can't use `sounddevice`, so we record via the browser with a little JavaScript.
Run this cell, allow mic access, click **Stop** when done — it transcribes automatically.

In [15]:
from IPython.display import Javascript, display
from google.colab import output
from base64 import b64decode
import io

RECORD_JS = """
const sleep = ms => new Promise(r => setTimeout(r, ms));
async function record() {
  const stream = await navigator.mediaDevices.getUserMedia({audio:true});
  const rec = new MediaRecorder(stream);
  const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  const stop = new Promise(r => {
    const btn = document.createElement('button');
    btn.textContent = 'Stop recording';
    document.body.appendChild(btn);
    btn.onclick = () => { rec.stop(); btn.remove(); r(); };
  });
  rec.start();
  await stop;
  await sleep(200);
  const blob = new Blob(chunks);
  const buf = await blob.arrayBuffer();
  const b64 = btoa(String.fromCharCode(...new Uint8Array(buf)));
  return b64;
}
"""

display(Javascript(RECORD_JS))
print("Click 'Allow' for mic, speak, then click 'Stop recording'...")
b64 = output.eval_js("record()")

with open("mic_recording.webm", "wb") as f:
    f.write(b64decode(b64))
print("Saved mic_recording.webm")

mic_result = transcriber.transcribe("mic_recording.webm")
print("\nYou said:", mic_result.text)

<IPython.core.display.Javascript object>

Click 'Allow' for mic, speak, then click 'Stop recording'...
Saved mic_recording.webm


/tmp/ipykernel_3731/3760931856.py:41: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(path, sr=target_sr, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)



You said: I don't know if it's 50% 50% 5 minutes


In [21]:
import os
for f in os.listdir():
    print(f)

.config
harvard.json
harvard.csv
harvard.vtt
harvard.txt
harvard.srt
harvard.wav
mic_recording.webm
sample_data


In [22]:
import os
for f in os.listdir():
    print(f)


.config
harvard.json
harvard.csv
harvard.vtt
harvard.txt
harvard.srt
harvard.wav
mic_recording.webm
sample_data


In [24]:
from google.colab import files
files.download('mic_recording.webm')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>